# LangGraph RAG 파이프라인

PDF 아웃라인 기반 전체 파이프라인 구현

```
사용자 입력
    ↓
[NODE 1] moderator1       : 정규식 → 민감정보 차단          (PDF 1번)
    ↓
[NODE 2] moderator2_intent: LLM → 의도 파악 + 확답 질문 차단 (PDF 2번)
    ↓
[NODE 3] retrieval        : 벡터 DB 검색 (메타데이터 필터)   (PDF 3번)
    ↓
[NODE 4] similarity_check : 유사도 임계치 판단              (PDF 4번)
    ↓
[NODE 5] generator        : 답변 생성 (Tavily Tool 조건부)   (PDF 5번)
    ↓
[NODE 6] formatter        : 답변 양식 정제                  (PDF 6번)
    ↓
[NODE 7] moderator3       : 확답 표현 필터                  (PDF 7번)
```

## 0. 환경변수 로드

`.env` 파일에서 `OPENAI_API_KEY`, `TAVILY_API_KEY`를 읽어온다.  
API 키를 코드에 직접 하드코딩하면 GitHub 유출 위험이 있으므로, `.env` + `.gitignore` 조합이 표준 관행이다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## 1. 라이브러리 임포트 & State 정의

### State란?
LangGraph에서 **모든 노드가 공유하는 데이터 구조**다.  
노드는 State를 받아서 → 변경된 부분만 딕셔너리로 반환한다.

### 왜 이 필드들인가?
| 필드 | 역할 | 설정 노드 |
|------|------|----------|
| `messages` | 대화 히스토리 누적 (add_messages 리듀서) | 전체 |
| `user_input` | 현재 사용자 입력 원문 | 초기값 |
| `metadata_filter` | RAG 검색 필터 | moderator2 |
| `retrieved_docs` | 검색된 문서 목록 | retrieval |
| `similarity_score` | 검색 최고 유사도 | retrieval |
| `is_sensitive` | 민감정보 포함 여부 | moderator1 |
| `is_definitive` | 확답 요구 질문 여부 | moderator2 |
| `needs_link` | 링크 요청 여부 (Tavily 트리거) | moderator2 |
| `final_answer` | 생성 → 정제 → 필터 거친 최종 답변 | generator~moderator3 |

In [ ]:
import re
import json
from typing import TypedDict, Annotated, List, Optional

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch


class State(TypedDict):
    messages: Annotated[List, add_messages]  # add_messages: 덮어쓰기 대신 누적(append)
    user_input: str
    metadata_filter: Optional[dict]
    retrieved_docs: Optional[List[dict]]
    similarity_score: Optional[float]
    is_sensitive: bool
    is_definitive: bool
    needs_link: bool
    final_answer: Optional[str]

## 2. LLM & Tool 초기화

- `gpt-4.1-mini`: PDF에서 "비용" 언급 → 성능 대비 저렴한 모델 선택  
- `TavilySearch(max_results=3)`: 링크 요청 시에만 사용. max_results를 3으로 제한해서 토큰 절약  
- `bind_tools`: LLM이 `tool_calls` 응답을 낼 수 있도록 도구 스키마를 등록

In [ ]:
# LLM 초기화 — 'provider:model' 형식으로 다양한 모델을 통일된 인터페이스로 사용
llm = init_chat_model('openai:gpt-4.1-mini')

# Tavily 검색 도구 — PDF 5번: 링크 요청 시에만 사용
tavily_tool = TavilySearch(max_results=3)

# Tool이 바인딩된 LLM — tool_calls 응답을 낼 수 있는 버전
llm_with_tools = llm.bind_tools([tavily_tool])

## 3. NODE 1: Moderator 1 — 민감정보 필터
**PDF 1번 구현**

### 왜 LLM 대신 정규식?
- 주민번호/계좌번호는 패턴이 수학적으로 고정 → 정규식으로 100% 탐지 가능  
- LLM 호출 = 비용 발생. 정규식 = 비용 0, 속도 마이크로초  
- PDF에서 "하드코딩으로 판별"이라고 명시

### 조건부 엣지 함수
`route_after_moderator1`이 `is_sensitive` 값을 보고 `END` 또는 `'moderator2_intent'`를 반환하면, LangGraph가 그 이름의 노드로 이동시킨다.

In [ ]:
# PDF 1번: 탐지할 민감정보 패턴 목록
SENSITIVE_PATTERNS = [
    (r'\d{6}-[1-4]\d{6}',                         "주민등록번호"),
    (r'\d{3}-\d{2}-\d{5}',                        "사업자등록번호"),
    (r'\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}', "카드번호"),
    (r'(?<!\d)\d{10,14}(?!\d)',                   "계좌번호"),
]


def moderator1(state: State) -> dict:
    """PDF 1번: 정규식으로 민감정보 탐지 → 감지 시 경고 메시지 반환 + is_sensitive=True"""
    user_input = state['user_input']

    for pattern, label in SENSITIVE_PATTERNS:
        if re.search(pattern, user_input):
            # PDF 1-B: 경고 메시지 (세션 종료가 아닌 이 턴만 차단)
            warning_msg = (
                f"⚠️ {label}로 보이는 정보가 포함되어 있습니다.\n"
                "민감한 정보가 포함되어있지 않나요? 다시 한 번 확인해주세요!\n\n"
                "예) 주민등록번호, 계좌번호와 같은 개인정보/민감정보는 "
                "보안상 입력하지 않도록 권장합니다."
            )
            return {
                'is_sensitive': True,
                'messages': [AIMessage(content=warning_msg)]
            }

    return {'is_sensitive': False}  # 정상 통과


def route_after_moderator1(state: State) -> str:
    """조건부 엣지 함수: is_sensitive 값에 따라 다음 노드 결정"""
    return END if state.get('is_sensitive', False) else 'moderator2_intent'

## 4. NODE 2: Moderator 2 — 의도 파악 (LLM)
**PDF 2번 구현**

### 한 번의 LLM 호출로 3가지를 동시 추출하는 이유
PDF에서 "비용" 강조 → LLM 호출 횟수를 최소화  
JSON 구조화 응답으로 `metadata_filter`, `is_definitive`, `needs_link`를 한 번에 획득

### JSON 파싱 안전처리
LLM이 가끔 ` ```json ... ``` ` 코드블록으로 감싸서 반환하는 경우도 있어서, `.strip('```json').strip('```')`으로 전처리 후 파싱한다.

In [ ]:
# PDF 2번: 의도 파악 프롬프트 — 짧게 유지 (비용 절감)
INTENT_SYSTEM_PROMPT = """\
사용자의 질문을 분석해서 아래 JSON만 반환하세요. 설명 없이.

분석 항목:
1. metadata_filter: 벡터 DB 검색에 사용할 필터
   - category: 질문 카테고리 (예: "법률", "의료", "금융", "일반" 등)
   - source: 참고할 출처 (파악 가능할 때만)
2. is_definitive: 확답을 요구하는 질문인지 여부
   - True 예시: "~이 맞나요?", "반드시 ~해야 하나요?", "~가 100% 맞죠?"
   - False 예시: "~에 대해 알려주세요", "~는 어떻게 하나요?"
3. needs_link: 사용자가 링크/출처/참고자료를 요청했는지 여부

응답 형식 (JSON만):
{
  "metadata_filter": {"category": "일반"},
  "is_definitive": false,
  "needs_link": false
}"""


def moderator2_intent(state: State) -> dict:
    """PDF 2번: LLM으로 의도 파악 → 메타데이터 추출 + 확답 질문 차단"""
    user_input = state['user_input']

    response = llm.invoke([
        SystemMessage(content=INTENT_SYSTEM_PROMPT),
        HumanMessage(content=user_input)
    ])

    # LLM이 코드블록으로 감쌀 경우를 대비한 전처리
    raw = response.content.strip().strip('```json').strip('```').strip()
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        # 파싱 실패 시 안전한 기본값 사용
        result = {"metadata_filter": {"category": "일반"}, "is_definitive": False, "needs_link": False}

    # PDF 2-C: 확답 질문 → Fallback END
    if result.get('is_definitive', False):
        fallback_msg = (
            "해당 질문은 명확한 확답을 드리기 어렵습니다.\n"
            "상황과 조건에 따라 답변이 달라질 수 있어요. "
            "더 구체적인 상황을 설명해주시면 더 도움이 될 수 있습니다."
        )
        return {
            'is_definitive': True,
            'metadata_filter': result.get('metadata_filter', {}),
            'needs_link': result.get('needs_link', False),
            'messages': [AIMessage(content=fallback_msg)]
        }

    return {
        'is_definitive': False,
        'metadata_filter': result.get('metadata_filter', {}),  # PDF 2-B: Retrieval 메타데이터
        'needs_link': result.get('needs_link', False)
    }


def route_after_moderator2(state: State) -> str:
    """조건부 엣지 함수: 확답 질문이면 END, 아니면 retrieval로"""
    return END if state.get('is_definitive', False) else 'retrieval'

## 5. NODE 3: Retrieval — 벡터 DB 검색
**PDF 3번 구현**

### 메타데이터 필터를 쓰는 이유
`moderator2`에서 추출한 `metadata_filter`를 `similarity_search`에 넘기면,  
전체 DB가 아닌 특정 카테고리/출처 범위 안에서만 검색한다. → 정확도 ↑, 속도 ↑

### 실제 연결 방법
아래 Mock 코드를 주석 처리된 실제 벡터스토어 코드로 교체하면 된다.  
FAISS, Chroma, Pinecone 등 모두 동일한 `similarity_search_with_score` 인터페이스를 제공한다.

In [ ]:
def retrieval(state: State) -> dict:
    """PDF 3번: 메타데이터 필터를 적용한 벡터 DB 검색"""
    user_input      = state['user_input']
    metadata_filter = state.get('metadata_filter', {})

    # ── 실제 벡터스토어 연결 시 아래 주석을 해제하고 Mock 코드를 삭제 ────────────
    # from langchain_community.vectorstores import FAISS
    # from langchain_openai import OpenAIEmbeddings
    #
    # vectorstore = FAISS.load_local("./vectorstore", OpenAIEmbeddings())
    # results = vectorstore.similarity_search_with_score(
    #     query=user_input,
    #     k=3,
    #     filter=metadata_filter          # PDF 2-B에서 선정한 메타데이터 필터
    # )
    # docs = [
    #     {"content": doc.page_content, "metadata": doc.metadata, "score": float(score)}
    #     for doc, score in results
    # ]
    # ─────────────────────────────────────────────────────────────────────────────

    # 개발/테스트용 Mock 데이터 (실제 벡터스토어 연결 후 삭제)
    docs = [
        {"content": "관련 문서 내용 1입니다.", "metadata": {"source": "doc1.pdf", "category": "일반"}, "score": 0.82},
        {"content": "관련 문서 내용 2입니다.", "metadata": {"source": "doc2.pdf", "category": "일반"}, "score": 0.65},
    ]

    top_score = docs[0]['score'] if docs else 0.0

    return {
        'retrieved_docs': docs,
        'similarity_score': top_score   # 유사도 판단에 사용 (NODE 4)
    }

## 6. NODE 4: Similarity Check — 유사도 임계치 판단
**PDF 4번 구현**

### 임계치 설계 (PDF 4-D)
```
score ≥ 0.70  →  높은 신뢰도  →  generator 진행
0.40 ~ 0.70  →  애매한 구간  →  꼬리질문 유도 (PDF 4-B)
score < 0.40  →  낮은 신뢰도  →  fallback END (PDF 4-A)
```

### CRAG 방식이란? (PDF 4-C)
**Corrective RAG**: 유사도가 낮을 때 포기하는 대신, Tavily 등 외부 검색으로  
보완 문서를 가져와서 재생성을 시도하는 방법론.  
추후 도입 시 `low_similarity_fallback` 분기에 `crag_web_search` 노드를 연결하면 된다.

In [ ]:
# PDF 4-D: 임계치 비교군 — 실제 데이터로 튜닝 필요
THRESHOLD_PASS   = 0.70   # 이 이상: 높은 신뢰도 → generator 진행
THRESHOLD_FOLLOW = 0.40   # 이 이상 ~ PASS 미만: 꼬리질문 유도
# THRESHOLD_FOLLOW 미만: 낮은 신뢰도 → fallback END

# PDF 4-C: CRAG (Corrective RAG) 도입 시 여기서 분기 추가
# 현재는 주석으로만 남겨둠 — 구현 복잡도 높음
# 추후: 'low_similarity' → 'crag_web_search' 노드 연결


def similarity_check(state: State) -> dict:
    """PDF 4번: 패스스루 노드 — 상태 변경 없이 라우팅 트리거 역할만 수행"""
    return {}  # similarity_score는 이미 retrieval에서 저장됨


def route_after_similarity(state: State) -> str:
    """조건부 엣지: 유사도 점수에 따라 3방향 분기"""
    score = state.get('similarity_score', 0.0)

    if score >= THRESHOLD_PASS:
        return 'generator'
    elif score >= THRESHOLD_FOLLOW:
        return 'follow_up'              # PDF 4-B: 꼬리질문 유도
    else:
        return 'low_similarity_fallback' # PDF 4-A: 정확한 답변 불가


def follow_up(state: State) -> dict:
    """PDF 4-B: 유사도 중간 구간 — 꼬리질문으로 사용자에게 추가 정보 유도"""
    msg = (
        "조금 더 정확한 답변을 드리기 위해 추가 정보가 필요합니다.\n"
        "질문을 좀 더 구체적으로 설명해 주시겠어요?\n"
        "예) 어떤 상황에서 발생한 문제인지, 관련된 조건이나 환경은 어떤지 등"
    )
    return {'messages': [AIMessage(content=msg)]}


def low_similarity_fallback(state: State) -> dict:
    """PDF 4-A: 유사도 낮음 — 정확한 답변 불가 안내"""
    msg = "저희는 해당 질문에 대한 정확한 답변을 가지고 있지 않아요. 다른 방식으로 질문해 주시면 더 잘 도움드릴 수 있습니다."
    return {'messages': [AIMessage(content=msg)]}

## 7. NODE 5: Generator — 답변 생성
**PDF 5번 구현**

### Tavily Tool 조건부 사용 (PDF 5번 Tool)
- `needs_link=True`일 때만 Tavily API 호출 → 불필요한 비용/지연 방지  
- PDF 5번-i: Tavily가 느릴 경우 대안 → 메타데이터에 미리 링크 포함

### Tool Call 흐름
```
llm_with_tools.invoke() → tool_calls 포함된 AIMessage 반환
    ↓
tavily_tool.invoke(args) → 검색 결과 반환
    ↓
ToolMessage로 결과를 메시지에 추가
    ↓
llm.invoke(전체 메시지) → 최종 답변 생성
```

In [ ]:
GENERATOR_SYSTEM_PROMPT = """\
당신은 정확하고 신뢰할 수 있는 AI 어시스턴트입니다.

규칙:
- 제공된 참고 문서를 기반으로 답변하세요.
- 문서에 없는 내용은 추측하거나 지어내지 마세요.
- 확답 표현(반드시, 무조건, 절대적으로, 100% 등)은 사용하지 마세요.
- 간결하고 명확하게 답변하세요."""


def generator(state: State) -> dict:
    """PDF 5번: 검색 문서 기반 답변 생성 + needs_link일 때만 Tavily Tool 사용"""
    user_input     = state['user_input']
    retrieved_docs = state.get('retrieved_docs', [])
    needs_link     = state.get('needs_link', False)

    # 검색된 문서를 컨텍스트로 조합
    context = "\n\n".join(
        f"[출처: {doc['metadata'].get('source', '알 수 없음')}]\n{doc['content']}"
        for doc in retrieved_docs
    )

    messages = [
        SystemMessage(content=GENERATOR_SYSTEM_PROMPT),
        HumanMessage(content=f"참고 문서:\n{context}\n\n질문: {user_input}")
    ]

    if needs_link:
        # PDF 5번 Tool: 링크 요청 시에만 Tavily 실행
        ai_response = llm_with_tools.invoke(messages)

        if hasattr(ai_response, 'tool_calls') and ai_response.tool_calls:
            tool_call   = ai_response.tool_calls[0]
            tool_result = tavily_tool.invoke(tool_call['args'])

            # 툴 결과를 대화에 추가 후 최종 응답 생성
            messages.append(ai_response)
            messages.append(ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_call['id']
            ))
            final_response = llm.invoke(messages)
        else:
            final_response = ai_response
    else:
        # 링크 불필요 → 일반 LLM 호출 (비용 절감)
        final_response = llm.invoke(messages)

    return {
        'final_answer': final_response.content,
        'messages': [final_response]
    }

## 8. NODE 6: Formatter — 답변 양식 정제
**PDF 6번 구현**

### 역할
generator 출력에서 AI 특유의 불필요한 서두("물론입니다", "좋은 질문입니다" 등)를 제거하고  
핵심 내용만 간결하게 구조화한다.

### 비용 최적화 팁
간단한 서두 제거는 LLM 없이 정규식으로 가능:  
```python
re.sub(r'^(물론입니다|네,|안녕하세요)[,.!]?\s*', '', text)
```
복잡한 구조화가 필요할 때만 LLM을 쓴다.

In [ ]:
FORMATTER_SYSTEM_PROMPT = """\
AI 응답을 사용자에게 보기 좋게 정제하세요.

규칙:
1. 불필요한 서두("물론입니다", "좋은 질문입니다" 등) 제거
2. 핵심 내용 위주로 구조화 (필요 시 번호 목록 사용)
3. 문장은 간결하게 (한 문장에 하나의 내용만)
4. 마크다운 형식 유지

수정된 응답만 반환하세요. 설명 없이.
"""

def formatter(state: State) -> dict:
    """PDF 6번: generator 출력을 사용자에게 노출하기 좋은 형태로 정제"""
    final_answer = state.get('final_answer', '')
    if not final_answer:
        return {}

    response = llm.invoke([
        SystemMessage(content=FORMATTER_SYSTEM_PROMPT),
        HumanMessage(content=final_answer)
    ])

    return {
        'final_answer': response.content,
        'messages': [AIMessage(content=response.content)]
    }



## 9. NODE 7: Moderator 3 — 확답 표현 필터
**PDF 7번 구현**

### 2단계 설계 이유
| 단계 | 방법 | 비용 | 목적 |
|------|------|------|------|
| 1단계 | 정규식 감지 | 0 | 확답 표현 존재 여부 빠르게 확인 |
| 2단계 | LLM 교체 | 발생 | 문맥을 유지하며 자연스럽게 표현 교체 |

확답 표현이 없는 경우(대부분의 정상 응답) → LLM 호출 완전 생략  
정규식만 쓰면 "반드시 → 일반적으로" 단순 치환 → 문맥이 어색해질 수 있음

In [ ]:
# 1차 정규식 감지 패턴
DEFINITIVE_PATTERNS = [
    r'반드시',
    r'무조건',
    r'절대적으로',
    r'확실히',
    r'틀림없이',
    r'100\s*%',
    r'무조건적',
]

MODERATOR3_SYSTEM_PROMPT = """\
아래 텍스트에서 확답을 주는 단정적 표현을 부드러운 표현으로 교체하세요.

교체 기준:
- "반드시" → "일반적으로"
- "무조건" → "대부분의 경우"
- "절대적으로" → "대체로"
- "확실히" → "일반적으로 보면"
- "틀림없이" → "보통은"
- "100%" → "대부분"

내용과 맥락은 유지하고, 단정적 표현만 교체하세요.
수정된 텍스트만 반환하세요."""


def moderator3(state: State) -> dict:
    """PDF 7번: 정규식 1차 감지 → 확답 표현 있을 때만 LLM으로 교체"""
    final_answer = state.get('final_answer', '')
    if not final_answer:
        return {}

    # 1차: 정규식 — 확답 표현 없으면 LLM 호출 생략 (비용 절감)
    has_definitive = any(re.search(p, final_answer) for p in DEFINITIVE_PATTERNS)

    if not has_definitive:
        return {'messages': [AIMessage(content=final_answer)]}

    # 2차: LLM — 문맥을 유지하며 자연스럽게 표현 교체
    response = llm.invoke([
        SystemMessage(content=MODERATOR3_SYSTEM_PROMPT),
        HumanMessage(content=final_answer)
    ])

    return {
        'final_answer': response.content,
        'messages': [AIMessage(content=response.content)]
    }

## 10. 그래프 구성 & 컴파일

모든 노드와 엣지를 연결해서 실행 가능한 그래프를 완성한다.

### 전체 흐름
```
START
  └→ moderator1 ──(민감정보)──→ END
       └→ moderator2_intent ──(확답 질문)──→ END
            └→ retrieval
                 └→ similarity_check
                      ├→ (score≥0.70) generator → formatter → moderator3 → END
                      ├→ (score 0.40~0.70) follow_up → END
                      └→ (score<0.40) low_similarity_fallback → END
```

In [ ]:
builder = StateGraph(State)

# ── 노드 등록 ──────────────────────────────────────────────────────
builder.add_node('moderator1',              moderator1)              # PDF 1번
builder.add_node('moderator2_intent',       moderator2_intent)       # PDF 2번
builder.add_node('retrieval',               retrieval)               # PDF 3번
builder.add_node('similarity_check',        similarity_check)        # PDF 4번 (라우터)
builder.add_node('follow_up',               follow_up)               # PDF 4-B
builder.add_node('low_similarity_fallback', low_similarity_fallback) # PDF 4-A
builder.add_node('generator',               generator)               # PDF 5번
builder.add_node('formatter',               formatter)               # PDF 6번
builder.add_node('moderator3',              moderator3)              # PDF 7번

# ── 엣지 연결 ─────────────────────────────────────────────────────
builder.add_edge(START, 'moderator1')

# moderator1: 민감정보 여부에 따라 분기
builder.add_conditional_edges(
    'moderator1', route_after_moderator1,
    {END: END, 'moderator2_intent': 'moderator2_intent'}
)

# moderator2: 확답 질문 여부에 따라 분기
builder.add_conditional_edges(
    'moderator2_intent', route_after_moderator2,
    {END: END, 'retrieval': 'retrieval'}
)

# retrieval → similarity_check (항상)
builder.add_edge('retrieval', 'similarity_check')

# similarity_check: 유사도 점수에 따라 3방향 분기
builder.add_conditional_edges(
    'similarity_check', route_after_similarity,
    {
        'generator':               'generator',
        'follow_up':               'follow_up',
        'low_similarity_fallback': 'low_similarity_fallback'
    }
)

# fallback 경로: 바로 END
builder.add_edge('follow_up',               END)
builder.add_edge('low_similarity_fallback', END)

# 정상 경로: generator → formatter → moderator3 → END
builder.add_edge('generator',  'formatter')
builder.add_edge('formatter',  'moderator3')
builder.add_edge('moderator3', END)

# 컴파일: 설계도 → 실행 가능한 그래프
graph = builder.compile()
graph  # 그래프 시각화

## 11. 실행 헬퍼 함수

`State`의 모든 필드를 초기값으로 채워서 그래프를 실행하고,  
마지막 AI 응답(`messages[-1].content`)만 반환하는 편의 함수.

In [ ]:
def run(user_input: str) -> str:
    """그래프를 실행하고 마지막 AI 응답 텍스트를 반환"""
    initial_state: State = {
        'messages':         [HumanMessage(content=user_input)],
        'user_input':       user_input,
        'metadata_filter':  None,
        'retrieved_docs':   None,
        'similarity_score': None,
        'is_sensitive':     False,
        'is_definitive':    False,
        'needs_link':       False,
        'final_answer':     None,
    }
    result = graph.invoke(initial_state)
    return result['messages'][-1].content

## 12. 테스트

4가지 케이스로 각 분기 경로를 검증한다.

In [ ]:
# 테스트 1: 정상 질문 → 전체 파이프라인 통과
# 예상 경로: moderator1 → moderator2 → retrieval → similarity_check → generator → formatter → moderator3
print("=== 테스트 1: 일반 질문 ===")
print(run("LangGraph에 대해 알려주세요."))

In [ ]:
# 테스트 2: 민감정보 포함 → moderator1에서 차단
# 예상 경로: moderator1 → END (경고 메시지)
print("=== 테스트 2: 민감정보 포함 ===")
print(run("내 주민번호는 900101-1234567이에요."))

In [ ]:
# 테스트 3: 확답 요구 질문 → moderator2에서 차단
# 예상 경로: moderator1 → moderator2 → END (확답 불가 안내)
print("=== 테스트 3: 확답 질문 ===")
print(run("이 방법이 100% 맞는 방법인가요?"))

In [ ]:
# 테스트 4: 링크 요청 → generator에서 Tavily Tool 사용
# 예상 경로: 전체 파이프라인 + Tavily API 호출
print("=== 테스트 4: 링크 요청 ===")
print(run("관련 참고 링크도 같이 알려주세요."))